# Train DistilBERT Food Problem Detector

Run this in Google Colab (Runtime → Run all). Output: `food_problem_detector.tflite`

Drop the `.tflite` file into `app/src/main/assets/` and rebuild.

In [ ]:
!pip install -q transformers torch datasets tensorflow

In [ ]:
# ═══ LABEL YOUR DATA HERE ═══
# 0 = safe (no coach intervention needed)
# 1 = problematic (trigger the coach)

training_data = [
    # Problematic examples (label = 1)
    ("burger", 650, 22, 1),
    ("fries", 400, 14, 1),
    ("pizza", 800, 21, 1),
    ("milkshake", 500, 20, 1),
    ("chocolate bar", 550, 23, 1),
    ("donut", 350, 22, 1),
    ("ice cream", 450, 21, 1),
    ("cookies", 600, 22, 1),
    ("chips", 300, 23, 1),
    ("bacon cheeseburger", 750, 22, 1),
    ("soda", 250, 15, 1),
    ("fried chicken", 700, 21, 1),
    ("instant noodles", 400, 23, 1),
    ("frozen pizza", 850, 22, 1),
    ("candy bar", 400, 22, 1),
    ("late night burger", 650, 23, 1),
    ("energy drink", 250, 22, 1),
    ("fried rice", 650, 22, 1),

    # Safe examples (label = 0)
    ("grilled chicken salad", 350, 13, 0),
    ("apple", 95, 14, 0),
    ("banana", 105, 14, 0),
    ("oatmeal", 300, 8, 0),
    ("greek yogurt", 150, 9, 0),
    ("almonds", 200, 16, 0),
    ("broccoli", 55, 12, 0),
    ("salmon", 350, 12, 0),
    ("avocado toast", 350, 9, 0),
    ("protein shake", 200, 8, 0),
    ("eggs", 300, 8, 0),
    ("turkey sandwich", 400, 13, 0),
    ("mixed vegetables", 150, 13, 0),
    ("rice and beans", 500, 14, 0),
    ("smoothie", 250, 10, 0),
    ("hummus and carrots", 200, 15, 0),
    ("chicken breast", 300, 12, 0),
]

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import numpy as np

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, data):
        self.samples = []
        for food, kcal, hour, label in data:
            text = f"{food} {kcal}cal at {hour}h"
            self.samples.append((text, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        tokens = tokenizer(text, padding='max_length', truncation=True,
                          max_length=64, return_tensors='pt')
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'labels': torch.tensor(label)
        }

dataset = FoodDataset(training_data)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
# Freeze all DistilBERT layers — only train the classification head
for param in model.distilbert.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=5e-4)

model.train()
for epoch in range(20):
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss = {total_loss/len(loader):.4f}")

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask']
        )
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == batch['labels']).sum().item()
        total += batch['labels'].size(0)
print(f"Accuracy: {100 * correct / total:.1f}%")

In [ ]:
!pip install -q onnx onnx2tf tf-keras

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Export to ONNX then TFLite
dummy_input = {
    'input_ids': torch.randint(0, 1000, (1, 64)),
    'attention_mask': torch.ones(1, 64)
}

torch.onnx.export(
    model, (dummy_input['input_ids'], dummy_input['attention_mask']),
    'model.onnx',
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch'}, 'attention_mask': {0: 'batch'}},
    opset_version=14
)

!onnx2tf -i model.onnx -o tflite_out -onw -qt per-tensor
!cp tflite_out/model_float32.tflite food_problem_detector.tflite
print("✅ Model exported: food_problem_detector.tflite")

In [ ]:
from google.colab import files
files.download('food_problem_detector.tflite')